Tools

Models can request to call tools that performs task such as  fetching data from the database , searching the web, running code. Tools are pairing of:
1. A schema including the name of tool, a description and/or argument definitions (often a Json schema.).
2. A function or coroutine to execute.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"]= os.getenv("OPENAI_API_KEY")
model= init_chat_model("gpt-4.1")
response = model.invoke("Why do parrots talk?")
response 

AIMessage(content='**Parrots "talk" because of their remarkable vocal mimicry abilities.** Here’s why and how:\n\n### Why do parrots "talk"?\n1. **Social Communication:**  \n   In the wild, parrots are very social animals. They use a wide range of sounds to communicate with their flock. Mimicking is a way to bond, establish territory, or alert others to danger.\n\n2. **Vocal Learning:**  \n   Parrots are among the few animals (along with some songbirds, dolphins, humans, etc.) that are *vocal learners*. This means they learn new sounds by listening and imitating.\n\n3. **Adaptation to Human Environment:**  \n   As pets, parrots consider their human caretakers as their "flock." They instinctively mimic sounds around them, including words, whistles, phone ringtones, and more—anything they hear often.\n\n4. **Attention-seeking:**  \n   Parrots quickly learn that saying certain words or making specific sounds gets reactions from people—laughter, talking, treats, or attention—which reinforc

In [3]:
from langchain.tools import tool

@tool

def get_weather(location:str)->str:
    """Get weather location at a location"""
    return f"Its sunny is the {location}"

model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What is the weather like in boston?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Tool: {tool_call['args']}")

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_8c02795173', 'id': 'chatcmpl-EXqogbEqg3NmQ5z1KTOolOyi4yoAt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a12bd2-2faa-7d61-a61d-1daa582ef66e-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'boston'}, 'id': 'call_DgSbxxNOyYpKIlAx2pva4Hr4', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 'output_tokens': 15, 'total_tokens': 68, 'input_token_details': {'audio': 0, 'cache_read

In [ ]:
### Tools execution loops

# Step 1 " model generate tool call"

messages = [{"role":"user", "content":"what is the weather in boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2 : Execute tools and collect the results

for tool_call in ai_msg.tool_calls:
    #Execute tools with generated argumets
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3 Pass result back to model for final response

final_response = model_with_tools.invoke(messages)
print(final_response.text)  


The weather in Boston is sunny. If you need more specific information, like temperature or forecast details, let me know!


In [5]:
messages


[{'role': 'user', 'content': 'what is the weather in boston?'},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 52, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_8c02795173', 'id': 'chatcmpl-EXr4TtsV3Oa3O189QGcRkxvROKtru', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a12be1-1dfe-74a2-bd42-bc1520b5edcc-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_6Awl05zF3xlP2m0mWrMOlHpD', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output